In [4]:
import os
import gzip
import pickle

### Settings

In [5]:
DIR_DATA = "data"
DIR_TEXTS = os.path.join(DIR_DATA, "texts")
FILE_METADATA = os.path.join(DIR_DATA, "metadata.csv")

In [8]:
# Load ngrams
NS = [2, 3, 4, 5]
NGRAM_SOURCE = "idx"
NGRAM_SOURCE_FILE = f"ngrams_{'-'.join(map(str, NS))}_{NGRAM_SOURCE}.pkl"
with open(os.path.join(DIR_DATA, NGRAM_SOURCE_FILE), "rb") as f:
    ngrams = pickle.load(f)
    
print(f"Loaded {', '.join(map(str, NS))}-grams ({NGRAM_SOURCE}) from {os.path.join(DIR_DATA, NGRAM_SOURCE_FILE)}.")

Loaded 2, 3, 4, 5-grams from data\ngrams_2-3-4-5_idx.pkl.


In [17]:
# Load dictionary mappings
INDEX_TO_TOKEN_FILE = f"index_to_token.pkl"
TOKEN_TO_INDEX_FILE = f"token_to_index.pkl"

with open(os.path.join(DIR_DATA, INDEX_TO_TOKEN_FILE), "rb") as f:
    idx_to_token = pickle.load(f)

with open(os.path.join(DIR_DATA, TOKEN_TO_INDEX_FILE), "rb") as f:
    token_to_idx = pickle.load(f)

print(f"Loaded index-to-token mapping from {os.path.join(DIR_DATA, INDEX_TO_TOKEN_FILE)}.")
print(f"Loaded token-to-index mapping from {os.path.join(DIR_DATA, TOKEN_TO_INDEX_FILE)}.")

Loaded index-to-token mapping from data\index_to_token.pkl.
Loaded token-to-index mapping from data\token_to_index.pkl.


In [10]:
sorted_ngrams = {
    n: sorted(ngrams[n].items(), key=lambda item: item[1], reverse=True) for n in NS
}
print(f"Sorted n-grams by frequency.")

Sorted n-grams by frequency.


In [15]:
print(f"Most common ngrams:")
for n in NS:
    print(f"  {n}-grams:")
    # n_gram is now a tuple of IDs
    for n_gram_ids, count in sorted_ngrams[n][:10]:
        words = [idx_to_token[i] for i in n_gram_ids]
        print(f"    \"{' '.join(words)}\": {count}")

Most common ngrams:
  2-grams:
    "of the": 86534
    "in the": 62280
    "to the": 41919
    ". i": 40771
    ". the": 39451
    ". he": 33729
    "and the": 27769
    "on the": 27336
    "the and": 26411
    "to be": 23204
  3-grams:
    ". he was": 6112
    ". it was": 5836
    "of the and": 4629
    ". it is": 3660
    "one of the": 3629
    "dau . of": 3503
    ". in the": 3139
    "out of the": 2958
    ". i am": 2932
    "the and the": 2868
  4-grams:
    "was killed in action": 1829
    "and was killed in": 1659
    "in action in the": 1551
    "no . battn .": 1483
    "lost in action in": 1428
    ". lost in action": 1427
    "action in the north": 1427
    "the north sept .": 1386
    "in the north sept": 1385
    "north sept . .": 1365
  5-grams:
    "and was killed in action": 1521
    "in action in the north": 1427
    "lost in action in the": 1426
    ". lost in action in": 1419
    "in the north sept .": 1385
    "the north sept . .": 1364
    "action in the north sept"

In [21]:
print(f"Most common ngrams excluding sentence punctuation:")
for n in NS:
    print(f"  {n}-grams:")
    i = 0
    # Assuming sorted_ngrams[n] is used here as it's defined in cell #VSC-7a12bfb2
    for n_gram_ids, count in sorted_ngrams[n]:
        # Convert IDs back to words
        n_gram_words = [idx_to_token[idx] for idx in n_gram_ids]
        
        # Check against punctuation set
        if any(token in {".", "!", "?"} for token in n_gram_words):
            continue
            
        print(f"    \"{' '.join(n_gram_words)}\": {count}")
        i += 1
        if i >= 10:
            break

Most common ngrams excluding sentence punctuation:
  2-grams:
    "of the": 86534
    "in the": 62280
    "to the": 41919
    "and the": 27769
    "on the": 27336
    "the and": 26411
    "to be": 23204
    "he was": 22126
    "at the": 22027
    "it was": 18659
  3-grams:
    "of the and": 4629
    "one of the": 3629
    "out of the": 2958
    "the and the": 2868
    "in the and": 2759
    "to the and": 2414
    "that he was": 2305
    "lost in action": 2212
    "killed in action": 2110
    "was killed in": 2081
  4-grams:
    "was killed in action": 1829
    "and was killed in": 1659
    "in action in the": 1551
    "lost in action in": 1428
    "action in the north": 1427
    "in the north sept": 1385
    "on the outbreak of": 1114
    "killed in action at": 1093
    "on the coast of": 923
    "at the same time": 911
  5-grams:
    "and was killed in action": 1521
    "in action in the north": 1427
    "lost in action in the": 1426
    "action in the north sept": 1353
    "was kille

In [18]:
print(f"Number of occurences of some bigrams:")
for bigram_words in [("the", "cat"), ("him", "!"), ("her", "!"), ("him", "?", ), ("her", "?")]:
    try:
        # Convert words to IDs before lookup
        bigram_ids = tuple(token_to_idx[word] for word in bigram_words)
        count = ngrams[len(bigram_words)].get(bigram_ids, 0)
    except KeyError:
        # If any word in bigram is not in vocab
        count = 0
    print(f"    {bigram_words}: {count}")

Number of occurences of some bigrams:
    ('the', 'cat'): 173
    ('him', '!'): 426
    ('her', '!'): 228
    ('him', '?'): 580
    ('her', '?'): 323


In [19]:
# Most common words that end sentences
print(f"Most common words that end sentences:")
sentence_endings_set = {".", "!", "?"}

ending_counts = {
    punctuation: {} for punctuation in sentence_endings_set
}

# The data structure in ngrams[2] is now {(id1, id2): count}
for bigram_ids, count in ngrams[2].items():
    # Convert punctuation ID back to string to check if it's an ending
    punctuation_str = idx_to_token[bigram_ids[1]]
    
    if punctuation_str in sentence_endings_set:
        word_str = idx_to_token[bigram_ids[0]]
        ending_counts[punctuation_str][word_str] = ending_counts[punctuation_str].get(word_str, 0) + count
        
for punctuation, words in ending_counts.items():
    print(f"  {punctuation}:")
    for word, count in sorted(words.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {word}: {count}")

Most common words that end sentences:
  .:
    .: 15232
    mr: 12150
    it: 9863
    him: 9376
    me: 6088
    mrs: 5667
    her: 4665
    the: 4531
    them: 4488
    battn: 4472
  ?:
    it: 1223
    you: 1108
    me: 898
    him: 580
    that: 538
    this: 455
    not: 450
    now: 368
    do: 362
    so: 359
  !:
    .: 1625
    me: 841
    it: 715
    you: 553
    him: 426
    ah: 413
    oh: 411
    alas: 365
    god: 342
    !: 316


In [20]:
# Most common words that start sentences by punctuation
print(f"Most common words that start sentences by punctuation:")
sentence_startings_set = {".", "!", "?"}

starting_counts = {
    punctuation: {} for punctuation in sentence_startings_set
}

# The data structure in ngrams[2] is now {(id1, id2): count}
for bigram_ids, count in ngrams[2].items():
    # Check punctuation (first element of bigram)
    punctuation_str = idx_to_token[bigram_ids[0]]
    
    if punctuation_str in sentence_startings_set:
        word_str = idx_to_token[bigram_ids[1]]
        starting_counts[punctuation_str][word_str] = starting_counts[punctuation_str].get(word_str, 0) + count
        
for punctuation, words in starting_counts.items():
    print(f"  {punctuation}:")
    for word, count in sorted(words.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    {word}: {count}")

Most common words that start sentences by punctuation:
  .:
    i: 40771
    the: 39451
    he: 33729
    and: 20956
    but: 16326
    it: 15692
    .: 15232
    she: 12068
    in: 11107
    a: 10250
  ?:
    i: 2810
    what: 1605
    and: 1463
    the: 1197
    he: 1193
    you: 1024
    it: 761
    is: 675
    but: 608
    how: 602
  !:
    i: 3516
    and: 1682
    the: 1561
    what: 1347
    you: 1339
    he: 1046
    but: 963
    how: 929
    it: 892
    my: 779
